# EHR Mamba Embedding on MIMIC-IV

This notebook trains the EHR Mamba model with the full paper §2.2 embedding on
MIMIC-IV in-hospital mortality prediction.

The pipeline uses three project files:
- `mortality_prediction_ehrmamba_mimic4.py` — `MIMIC4EHRMambaMortalityTask`, `LabQuantizer`, `collate_ehr_mamba_batch`
- `ehrmamba_embedding.py` — `EHRMambaEmbedding` (7-component §2.2 / Eq. 1 embedding)
- `ehrmamba_v2.py` — `EHRMamba` model


In [8]:
from pyhealth.datasets import MIMIC4EHRDataset, split_by_sample
from pyhealth.trainer import Trainer
from pyhealth.tasks.mortality_prediction_ehrmamba_mimic4 import (
    MIMIC4EHRMambaMortalityTask,
    LabQuantizer,
    collate_ehr_mamba_batch,
)
from pyhealth.models.ehrmamba_v2 import EHRMamba
from torch.utils.data import DataLoader
import torch


ModuleNotFoundError: No module named 'pyhealth'

## 1. Load MIMIC-IV Dataset

Set `DEV_MODE = False` to use the full dataset.


In [ ]:
DATA_ROOT = r'C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\data'

dataset = MIMIC4EHRDataset(
    root=DATA_ROOT,
    tables=['patients', 'admissions', 'procedures_icd', 'prescriptions', 'labevents'],
    dev=True,
)


## 2. Fit Lab Quantizer and Set Task

`LabQuantizer` bins continuous lab values into 5 quantile tokens per `itemid`
(paper Appendix B). `MIMIC4EHRMambaMortalityTask` builds the §2.1 token sequence:
`[CLS] [VS] events [VE] [REG] [W?] ...` with in-hospital mortality as the label.


In [ ]:
# Fit LabQuantizer on all patients (fit only on train split to avoid leakage in production)
lab_quantizer = LabQuantizer(n_bins=5)
lab_quantizer.fit_from_patients(list(dataset.patients.values()))

task = MIMIC4EHRMambaMortalityTask(
    min_age=18,
    lab_quantizer=lab_quantizer,
    tables=('procedures_icd', 'prescriptions', 'labevents'),
)
sample_dataset = dataset.set_task(task)
train_dataset, val_dataset, test_dataset = split_by_sample(
    sample_dataset, ratios=[0.7, 0.1, 0.2]
)


## 3. Create Data Loaders

`collate_ehr_mamba_batch` right-pads variable-length sequences and stacks the six
EHR Mamba tensor fields (`input_ids`, `token_type_ids`, `time_stamps`, `ages`,
`visit_orders`, `visit_segments`) into `(B, L_max)` tensors.


In [ ]:
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True,
    collate_fn=collate_ehr_mamba_batch,
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)


## 4. Initialize EHRMamba Model

`use_ehr_mamba_embedding=True` replaces PyHealth's default embedding with
`EHRMambaEmbeddingAdapter(EHRMambaEmbedding(...))` — the full 7-component
Equation 1 embedding from §2.2.


In [ ]:
model = EHRMamba(
    dataset=sample_dataset,
    embedding_dim=128,
    num_layers=2,
    state_size=16,
    conv_kernel=4,
    dropout=0.1,
    use_ehr_mamba_embedding=True,
)
trainer = Trainer(model=model, metrics=['roc_auc', 'pr_auc'])
print(model)


## 5. Train

In [ ]:
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    monitor='roc_auc',
    optimizer_params={'lr': 1e-4},
)


## 6. Evaluate on Test Set

In [ ]:
trainer.evaluate(test_loader)


## 7. Get Patient Embeddings

Pass `embed=True` to retrieve pooled patient representations from the last
Mamba block. These can be used for downstream tasks or visualisation.


In [ ]:
model.eval()
with torch.no_grad():
    batch = next(iter(test_loader))
    output = model(**batch, embed=True)
    embeddings = output['embed']
    print(f'Patient embeddings shape: {embeddings.shape}')
